# Modelio testavimas ir validacija su Python (OLS regresija)

Naudojamas `tips` duomenų rinkinys iš `seaborn`. Pateikiami pagrindiniai žingsniai, kaip vertinti modelio kokybę, valdyti overfitting/underfitting riziką ir taikyti cross validation.

Pastaba: šiame faile pateikiami tiek paprasti pavyzdžiai su vienu split, tiek cross validation. Pavyzdžiai orientuoti į prognozavimo kokybės vertinimą ir aiškią interpretaciją.


In [1]:
# Bibliotekos
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import statsmodels.api as sm

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# Duomenys
tips = sns.load_dataset("tips").dropna().copy()
tips.head()


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


## Model scoring žingsniai (bendras algoritmas)

1. Duomenų paruošimas (trūkstamos reikšmės, kategoriniai kintamieji, dtypes).
2. Duomenų padalijimas į train ir test (dažnai papildomai į validation).
3. Modelio apmokymas (fit) ant train.
4. Prognozės (predict) ant train/validation/test.
5. Metrikų skaičiavimas (pvz., R², MAE, MSE/RMSE).
6. Modelio parinkimas ir tuning pagal validation arba cross validation rezultatus.
7. Galutinis įvertinimas ant test (test naudojamas vieną kartą, pabaigoje).

Dažnos klaidos:
- test rinkinio naudojimas tuning metu (per optimistiškas įvertinimas);
- neatskirtas validation (modelis „derinamas“ ant test);
- neteisingai sukurti dummy kintamieji (bool/object tipai sukelia `ValueError`).


## Duomenų paruošimas ir dummy kintamieji

Modeliuojamas target: `tip`.
Naudojami požymiai: `total_bill`, `size`, `day`, `time`, `sex`, `smoker`.

Kategoriniai kintamieji paverčiami į dummy stulpelius. `drop_first=True` sumažina pilno multikolinerumo riziką. Po `get_dummies` stulpeliai konvertuojami į `float`, kad `statsmodels` ir `sklearn` dirbtų stabiliai.


In [2]:
# X ir y
X_raw = tips[["total_bill", "size", "day", "time", "sex", "smoker"]].copy()
y = tips["tip"].astype(float)

# Dummy + float (kritiška dėl ValueError)
X = pd.get_dummies(X_raw, drop_first=True).astype(float)

X.head(), X.dtypes.head()


(   total_bill  size  day_Fri  day_Sat  day_Sun  time_Dinner  sex_Female  \
 0       16.99   2.0      0.0      0.0      1.0          1.0         1.0   
 1       10.34   3.0      0.0      0.0      1.0          1.0         0.0   
 2       21.01   3.0      0.0      0.0      1.0          1.0         0.0   
 3       23.68   2.0      0.0      0.0      1.0          1.0         0.0   
 4       24.59   4.0      0.0      0.0      1.0          1.0         1.0   
 
    smoker_No  
 0        1.0  
 1        1.0  
 2        1.0  
 3        1.0  
 4        1.0  ,
 total_bill    float64
 size          float64
 day_Fri       float64
 day_Sat       float64
 day_Sun       float64
 dtype: object)

## Data splitting

Dažniausi split variantai:
- train/test: greitas patikrinimas, bet tuning rizikuoja „nutekėti“ į test;
- train/validation/test: geresnė praktika, kai reikia rinktis modelį ar hyperparametrus;
- cross validation: robustiškesnis įvertinimas, kai duomenų nėra daug.

Šiame pavyzdyje:
- test dydis: 20 %
- validation dydis: 20 % nuo likusios dalies (t. y. 16 % nuo visų duomenų)


In [3]:
# 1) Train/Test split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# 2) Train/Validation split iš train_full
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=42
)

X_train.shape, X_val.shape, X_test.shape


((156, 8), (39, 8), (49, 8))

## Overfitting ir underfitting

- Underfitting: modelis per paprastas, blogai tinka tiek train, tiek test (aukšta paklaida).
- Overfitting: modelis per sudėtingas, puikiai tinka train, bet blogėja ant validation/test (aukšta variacija).

Praktinis indikatorius:
- jei train metrika labai gera, o validation/test ryškiai prastesnė, tikėtinas overfitting;
- jei visur prasta, tikėtinas underfitting.

Bias-variance tradeoff:
- didinant modelio sudėtingumą, bias mažėja, bet variance dažnai didėja;
- tikslas yra rasti balansą, kuris geriausiai generalizuoja į naujus duomenis.


## Bazinis modelis ir scoring

Pateikiamas OLS (statsmodels) modelis interpretacijai ir `sklearn` modelis scoring/cross validation patogumui.

Naudojamos metrikos:
- R²: paaiškintos variacijos dalis (aukštesnė – geriau).
- MAE: vidutinė absoliuti klaida (mažesnė – geriau).
- RMSE: kvadratinė klaida (jautresnė outlieriams).


In [4]:
# OLS (statsmodels) ant train
ols_train = sm.OLS(y_train, sm.add_constant(X_train)).fit()
ols_train.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                    tip   R-squared:                       0.409
Model:                            OLS   Adj. R-squared:                  0.377
Method:                 Least Squares   F-statistic:                     12.72
Date:                Tue, 03 Feb 2026   Prob (F-statistic):           8.46e-14
Time:                        11:27:02   Log-Likelihood:                -234.57
No. Observations:                 156   AIC:                             487.1
Df Residuals:                     147   BIC:                             514.6
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const           0.7900      0.348      2.273      0.024       0.103       1.477
total_bill      0.0880      0.013      6.580      0.000       0.062       0.114
size            0.2145      0.129      1.667      0.098      -0.040       0.469
day_Fri        -0.0918      0.470     -0.195      0.846      -1.021       0.838
day_Sat        -0.3987      0.595     -0.670      0.504      -1.575       0.778
day_Sun        -0.1857      0.594     -0.313      0.755      -1.359       0.988
time_Dinner     0.2472      0.559      0.443      0.659      -0.857       1.351
sex_Female     -0.0875      0.196     -0.446      0.656      -0.475       0.300
smoker_No       0.1296      0.194      0.666      0.506      -0.255       0.514
==============================================================================
Omnibus:                       15.600   Durbin-Watson:                   1.755
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               24.764
Skew:                           0.524   Prob(JB):                     4.19e-06
Kurtosis:                       4.646   Cond. No.                         260.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

### Train modelio rezultatų interpretacija

Modelis paaiškina apie 41 % arbatpinigių (`tip`) variacijos train duomenyse (R² = 0.409). F-statistikos p reikšmė yra labai maža, todėl modelis kaip visuma yra statistiškai reikšmingas.

Reikšmingiausias kintamasis yra `total_bill`. Jo koeficientas teigiamas ir statistiškai reikšmingas, todėl didėjant sąskaitos sumai, vidutiniai arbatpinigiai didėja. Tai yra pagrindinis arbatpinigių dydį paaiškinantis veiksnys.

Kintamasis `size` turi teigiamą poveikį, tačiau nėra statistiškai reikšmingas. Kiti kategoriniai kintamieji (`day`, `time`, `sex`, `smoker`) taip pat neturi reikšmingo papildomo poveikio, kai įvertinama sąskaitos suma.

Durbin–Watson statistika yra arti 2, todėl ryškios rezidualų autokoreliacijos neaptinkama. Normalumo testai rodo nukrypimus nuo normalumo, tačiau tai nėra kritiška prognozavimo kontekste.

Išvada: train duomenyse arbatpinigių dydį daugiausia lemia sąskaitos suma, o kiti nagrinėti požymiai turi ribotą paaiškinamąją vertę.


In [5]:
# Scoring funkcija (vienoje vietoje)
def score_regression(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    return {"R2": r2, "MAE": mae, "RMSE": rmse}

# Prognozės
pred_train = ols_train.predict(sm.add_constant(X_train))
pred_val = ols_train.predict(sm.add_constant(X_val))
pred_test = ols_train.predict(sm.add_constant(X_test))

scores = pd.DataFrame([
    {"split": "train", **score_regression(y_train, pred_train)},
    {"split": "validation", **score_regression(y_val, pred_val)},
    {"split": "test", **score_regression(y_test, pred_test)},
])

scores


,split,R2,MAE,RMSE
0,train,0.409016,0.794876,1.088412
1,validation,0.596985,0.702176,0.906447
2,test,0.426440,0.684422,0.846719


### Kaip skaityti train/validation/test skirtumus

- Jei `train` R² yra ryškiai didesnis nei `validation/test`, modelis tikėtina per daug prisitaiko prie train (overfitting).
- Jei metrikos panašios per visus split, generalizacija yra stabilesnė.
- Test rezultatas interpretuojamas kaip galutinis „tikėtinas“ veikimas naujuose duomenyse.

Dažna klaida:
- lyginti tik R², ignoruojant MAE/RMSE; praktikoje absoliuti klaida dažnai geriau atspindi verslo nuostolius.


### Train / Validation / Test metrikų interpretacija

Modelis pasiekia vidutinį paaiškinamumą train duomenyse (R² ≈ 0.41). Validation rinkinyje R² yra didesnis, o klaidos (MAE, RMSE) mažesnės, kas rodo, kad šis validation split yra „palankesnis“ modeliui ir nėra tinkamas vienintelis sprendimo pagrindas.

Test rinkinyje metrikos yra artimos train rezultatams. Tai rodo, kad modelis **neperfitina** ir pakankamai stabiliai generalizuoja į naujus duomenis.

Išvada: modelis nėra per daug sudėtingas, neturi ryškaus overfitting požymių, o galutinį veikimą geriausia vertinti pagal test rinkinio metrikas.


## Model tuning (paprastas pavyzdys)

Tuning reiškia modelio parinkimą, kuris geriausiai veikia validation (arba cross validation). Šiame pavyzdyje naudojamas paprastas „sudėtingumo“ didinimas pridedant polinominį terminą `total_bill²`.

Svarbu: polinominiai terminai naudojami hierarchiškai, todėl kartu paliekamas ir `total_bill`.


In [6]:
# Sukuriamas polinominis terminas
X_poly = X.copy()
X_poly["total_bill_sq"] = X_poly["total_bill"] ** 2

# Split analogiškai (tą patį random_state, kad būtų palyginama)
Xp_train_full, Xp_test, yp_train_full, yp_test = train_test_split(
    X_poly, y, test_size=0.20, random_state=42
)
Xp_train, Xp_val, yp_train, yp_val = train_test_split(
    Xp_train_full, yp_train_full, test_size=0.20, random_state=42
)

ols_poly = sm.OLS(yp_train, sm.add_constant(Xp_train)).fit()

poly_scores = pd.DataFrame([
    {"split": "train", **score_regression(yp_train, ols_poly.predict(sm.add_constant(Xp_train)))},
    {"split": "validation", **score_regression(yp_val, ols_poly.predict(sm.add_constant(Xp_val)))},
    {"split": "test", **score_regression(yp_test, ols_poly.predict(sm.add_constant(Xp_test)))},
])

poly_scores


,split,R2,MAE,RMSE
0,train,0.409829,0.798368,1.087663
1,validation,0.605435,0.708489,0.896894
2,test,0.399694,0.711690,0.866236


### Tuning sprendimas pagal validation

Jei polinominis modelis pagerina validation metrikas, jis gali būti pasirenkamas kaip geresnis variantas. Jei validation negerėja (ar blogėja), terminas laikomas nereikalingu, nes tik apsunkina interpretaciją ir gali didinti variaciją.

Gerosios praktikos:
- tuning daryti pagal validation arba cross validation, ne pagal test;
- test palikti galutiniam vienkartiniam įvertinimui.


### Polinominio modelio (su `total_bill²`) metrikų interpretacija

Pridėjus polinominį terminą `total_bill²`, train metrikos beveik nepasikeitė, o test rinkinyje R² net šiek tiek sumažėjo, palyginti su baziniu modeliu. Tai rodo, kad papildomas terminas **nepagerino generalizacijos**.

Validation rinkinyje R² yra aukštesnis, tačiau šis pagerėjimas nėra patvirtinamas test rinkinyje, todėl jis laikytinas atsitiktinio split efekto rezultatu.

Išvada: kadangi polinominis terminas **šiuo** atveju *didina modelio sudėtingumą, bet nesuteikia realios naudos*, todėl bazinis tiesinis modelis yra tinkamesnis pasirinkimas.


In [7]:
# Bazinio ir polinominio modelio palyginimas (validation)
compare = pd.DataFrame({
    "modelis": ["bazinis OLS", "polinominis OLS"],
    "R2_val": [scores.loc[scores["split"]=="validation","R2"].iloc[0],
              poly_scores.loc[poly_scores["split"]=="validation","R2"].iloc[0]],
    "MAE_val": [scores.loc[scores["split"]=="validation","MAE"].iloc[0],
               poly_scores.loc[poly_scores["split"]=="validation","MAE"].iloc[0]],
    "RMSE_val": [scores.loc[scores["split"]=="validation","RMSE"].iloc[0],
                poly_scores.loc[poly_scores["split"]=="validation","RMSE"].iloc[0]],
})
compare


,modelis,R2_val,MAE_val,RMSE_val
0,bazinis OLS,0.596985,0.702176,0.906447
1,polinominis OLS,0.605435,0.708489,0.896894


## Cross validation (k-fold)

Cross validation sumažina priklausomybę nuo vieno split. Duomenys padalinami į `k` dalis (folds), modelis apmokomas `k` kartų, kiekvieną kartą validuojant ant skirtingo fold.

Privalumai:
- stabilesnis įvertinimas, ypač kai duomenų mažai;
- mažesnė rizika, kad vienas „nesėkmingas“ split iškreips rezultatą.

Trūkumai:
- didesnė skaičiavimo kaina;
- tuning procesas sudėtingesnis (reikia CV + pasirinkimų palyginimo).

Šiame pavyzdyje naudojamas `sklearn` `LinearRegression` ir `cross_val_score`.


In [8]:
# Sklearn modelis (linijinis)
lr = LinearRegression()

# KFold su shuffle, kad eilučių tvarka neįtakotų rezultatų
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# R² per CV
cv_r2 = cross_val_score(lr, X, y, cv=kf, scoring="r2")
cv_r2, cv_r2.mean(), cv_r2.std()


(array([0.43730182, 0.09225183, 0.56146197, 0.50059724, 0.32369906]),
 0.38306238247895985,
 0.1653345431472497)

### K-fold cross validation rezultatų interpretacija

Pateiktos penkios reikšmės yra R² rezultatai iš kiekvieno k-fold padalinimo. Jos rodo, kaip modelis veikia skirtinguose duomenų pogrupiuose.

Vidutinė R² reikšmė yra apie 0.38, o standartinis nuokrypis yra apie 0.17. Tai reiškia, kad modelio paaiškinamoji galia yra vidutinė, tačiau rezultatai tarp skirtingų fold reikšmingai svyruoja.

Didelė sklaida tarp fold rodo, kad modelio veikimas yra jautrus konkrečiam duomenų padalinimui. Tai būdinga nedideliems duomenų rinkiniams ir gana paprastiems modeliams.

Išvada: cross validation suteikia realesnį vaizdą apie modelio stabilumą nei vienas train/test split. Vertinant modelį, svarbu atsižvelgti ne tik į vidutinę metriką, bet ir į jos sklaidą.


In [9]:
# MAE ir RMSE per CV (sklearn grąžina neigiamus dydžius, todėl dedamas minusas)
cv_mae = -cross_val_score(lr, X, y, cv=kf, scoring="neg_mean_absolute_error")
cv_rmse = -cross_val_score(lr, X, y, cv=kf, scoring="neg_root_mean_squared_error")

pd.DataFrame({
    "metric": ["R2", "MAE", "RMSE"],
    "cv_mean": [cv_r2.mean(), cv_mae.mean(), cv_rmse.mean()],
    "cv_std": [cv_r2.std(), cv_mae.std(), cv_rmse.std()],
})


,metric,cv_mean,cv_std
0,R2,0.383062,0.165335
1,MAE,0.765403,0.108078
2,RMSE,1.044335,0.136131


### Cross validation metrikų santrauka

Vidutiniai k-fold cross validation rezultatai ir jų sklaida:

Vidutinė R² reikšmė (~0.38) rodo, kad modelis paaiškina apie 38 % arbatpinigių variacijos skirtinguose duomenų pogrupiuose. Standartinis nuokrypis (~0.17) yra pakankamai didelis, todėl modelio paaiškinamoji galia nėra stabili tarp fold.

MAE vidurkis (~0.77) parodo, kad vidutinė prognozės paklaida yra apie 0.77 piniginio vieneto. RMSE (~1.04) yra didesnė, nes ši metrika labiau reaguoja į didesnes klaidas ir outlierius.

Išvada: modelio prognozavimo tikslumas yra vidutinis, o rezultatai tarp skirtingų fold pastebimai svyruoja. Cross validation rezultatai suteikia realesnį ir atsargesnį modelio veikimo įvertinimą nei vienas train/test padalinimas.


## Simple vs cross validation

Paprastas train/validation/test tinka, kai:
- duomenų daug ir vieno split rezultatas stabilus;
- reikia aiškaus test rinkinio galutiniam įvertinimui;
- tuning daromas tik su validation.

Cross validation tinka, kai:
- duomenų mažiau ir vienas split gali būti atsitiktinai sėkmingas/nesėkmingas;
- norima matyti ne tik vidurkį, bet ir rezultatų sklaidą;
- lyginami keli variantai (skirtingi požymiai, transformacijos ar modeliai).

Gerosios praktikos:
- jei tuning daromas su cross validation, test vis tiek paliekamas pabaigai;
- pateikiant CV rezultatą verta rodyti vidurkį ir standartinį nuokrypį.


## Dažnos klaidos ir gerosios praktikos (santrauka)

Dažnos klaidos:
- `pd.get_dummies()` paliekami `bool`/`object` stulpeliai, vėliau gaunamas `ValueError` su OLS.
- Test rinkinys naudojamas modelio pasirinkimui (nutekėjimas į test).
- Metrikų interpretacija be verslo konteksto (pvz., MAE kaip „vidutinė piniginė paklaida“).
- Cross validation naudojamas be `shuffle=True`, kai duomenys yra surikiuoti (dirbtinės priklausomybės).

Gerosios praktikos:
- po dummy kintamųjų kūrimo naudoti `.astype(float)`.
- turėti validation tuning etapui arba naudoti cross validation.
- test naudoti vieną kartą, galutiniam įvertinimui.
- vertinti bent dvi metrikas: R² ir MAE/RMSE.
